In [1]:
import os

os.chdir("..")

In [46]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
from docx import Document
import pdfplumber
import re
from io import BytesIO



In [47]:
def gather_links(config):
    os.makedirs(config.get('output_folder'), exist_ok=True)

    session = requests.Session()
    if config.get('User-Agent'):
        session.headers.update({"User-Agent": config.get('User-Agent')})

    page = session.get(config.get('base_url'))
    page.raise_for_status()

    # parse the page with BeautifulSoup
    soup = BeautifulSoup(page.text, 'html.parser')

    # find all desired doc links and remove duplicates
    article_links = list(set([
        urljoin(config.get('base_url'), href)
        for href in (a.get("href") for a in soup.find_all("a"))
        if href and any(href.lower().endswith(ext) for ext in config.get('doc_types_to_download'))
    ]))

    return article_links



In [48]:
def get_file_type(filename):
    if filename.lower().endswith(".pdf"):
        return "pdf"
    if filename.lower().endswith(".docx"):
        return "docx"
    return None

def extract_docx_text(url):
    time.sleep(1)
    response = requests.get(url)
    response.raise_for_status()

    file_obj = BytesIO(response.content)
    doc = Document(file_obj)

    return "\n".join(para.text for para in doc.paragraphs)


def extract_pdf_text(url):
    time.sleep(1)
    response = requests.get(url)
    response.raise_for_status()

    file_obj = BytesIO(response.content)

    text = ""
    with pdfplumber.open(file_obj) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


def process_links(article_links : list):
    all_text = []

    for link in article_links:
        print(f"Processing: {link}")
        try:

            filetype = get_file_type(link)

            if filetype == "pdf":
                text = extract_pdf_text(link)
            elif filetype == "docx":
                text = extract_docx_text(link)
            else:
                print(f"Skipping unknown file type: {link}")
                continue

            all_text.append(text)
        except Exception as e:
            print(f"Error processing {link}: {e}")
    
    return all_text



In [49]:
def remove_text_before_colon(docs: list[str]) -> str:
    cleaned_docs = []
    for text in docs:
        idx = text.find(" : ")
        if idx == -1:
            cleaned_docs.append(text)
        else:
            cleaned_docs.append(text[idx + len(" : "):])
    
    return cleaned_docs




In [50]:
def clean_text(text :list[str]):
    cleaned_docs = []
    for s in text:
        # Remove escape sequences like \n, \t
        s = s.replace("\n", "").replace("\t", "")

        # Collapse multiple spaces into one
        s = re.sub(r"\s+", " ", s)

        # Strip leading/trailing whitespace
        s = s.strip()

        cleaned_docs.append(s)
    return cleaned_docs



In [51]:
config =   {
        'base_url' : 'https://www.uspcak9.com/k9-training-articles',
        'output_folder' : 'rag',
        'output_filename' : 'prepared_docs.json',
        'doc_types_to_download' : ['.pdf', '.docx'],
        'User-Agent' : 'Mozilla/5.0',        # Comment line for a generic session

    }

In [52]:
links = gather_links(config)
 

In [53]:
for link in links:
    print(link)

https://www.uspcak9.com/assets/docs/The%20Urgent%20Need%20of%20Integrated%20Drone%20K-9%20Training%20%281%29.docx
https://www.uspcak9.com/assets/docs/You%20are%20Now%20a%20Canine%20Supervisor%20-%20WhiteS%20comment%20--%2020240123%20%281%29.docx
https://uspcak9.memberclicks.net/assets/Resource/The%20Start%20is%20the%20Most%20Difficult%20Part.pdf
https://www.uspcak9.com/assets/docs/Basic%20Police%20Working%20Dog%20Training%20Academy.docx
https://uspcak9.memberclicks.net/assets/Resource/Body_Language.pdf
https://uspcak9.memberclicks.net/assets/ForumArticles/Trainers%20Corner_Testing%20for%20a%20proper%20Police%20dog_Courier_November_2018.pdf
https://uspcak9.memberclicks.net/assets/Resource/USPCAHeat.pdf
https://www.uspcak9.com/assets/docs/CONTROLLED%20CHAOS.docx
https://uspcak9.memberclicks.net/assets/docs/A%20guide%20to%20effectively%20presenting%20body.docx
https://www.uspcak9.com/assets/Courier/HWEmergprotocol.pdf
https://www.uspcak9.com/assets/docs/K-9%20Unit%20Fundraising%20Article%

In [55]:
print(len(links))
links[50]

59


'https://www.uspcak9.com/assets/docs/Clarity%20in%20Police%20Canine%20Training.docx'

In [56]:
raw_text = process_links(links)


Processing: https://www.uspcak9.com/assets/docs/The%20Urgent%20Need%20of%20Integrated%20Drone%20K-9%20Training%20%281%29.docx
Processing: https://www.uspcak9.com/assets/docs/You%20are%20Now%20a%20Canine%20Supervisor%20-%20WhiteS%20comment%20--%2020240123%20%281%29.docx
Processing: https://uspcak9.memberclicks.net/assets/Resource/The%20Start%20is%20the%20Most%20Difficult%20Part.pdf
Processing: https://www.uspcak9.com/assets/docs/Basic%20Police%20Working%20Dog%20Training%20Academy.docx
Processing: https://uspcak9.memberclicks.net/assets/Resource/Body_Language.pdf
Processing: https://uspcak9.memberclicks.net/assets/ForumArticles/Trainers%20Corner_Testing%20for%20a%20proper%20Police%20dog_Courier_November_2018.pdf
Processing: https://uspcak9.memberclicks.net/assets/Resource/USPCAHeat.pdf
Processing: https://www.uspcak9.com/assets/docs/CONTROLLED%20CHAOS.docx
Processing: https://uspcak9.memberclicks.net/assets/docs/A%20guide%20to%20effectively%20presenting%20body.docx
Processing: https://ww

In [57]:
raw_text

['\nDavid C.A. Heptinstall\nSergeant, Retired\nOfficer safety remains a central concern in modern law enforcement as agencies face increasingly complex and dynamic operational environments. This paper argues that the integration of Canine and sUAS units through structured joint training is a critical strategy for enhancing officer safety and operational effectiveness. Drawing on practical law enforcement experience, the paper examines how siloed team structures can undermine communication, coordination, and situational awareness during high-risk incidents. It contends that joint training fosters shared tactics, common language, and mutual trust among units, leading to more cohesive responses in real-world operations. The paper concludes that intentional, integrated training programs are not merely beneficial but essential for agencies seeking to reduce risk, improve decision-making, and protect officers in the field.  This is based on years of experience as a Patrol K-9 Handler, and la